# Amazon Bedrock Foundation Model Benchmark

## Objective

Evaluate multiple Amazon Bedrock foundation models for a financial-services customer assistant.

Models evaluated:

- Amazon Nova Micro
- Amazon Nova Lite
- Amazon Nova Pro

Metrics:

- Response quality
- Latency
- Input/output token usage
- Cost per request
- Throughput

The goal is to determine which model provides the best balance of quality, performance, and cost.

In [41]:
import boto3
import time
import pandas as pd

from concurrent.futures import ThreadPoolExecutor

session = boto3.Session(
    profile_name="neotrix"
)

bedrock = session.client(
    "bedrock-runtime",
    region_name="us-east-1"
)

In [20]:
models = [
    "amazon.nova-micro-v1:0",
    "amazon.nova-lite-v1:0",
    "amazon.nova-pro-v1:0"
]


MODEL_PRICING = {
    "amazon.nova-micro-v1:0": {
        "input": 0.000035,
        "output": 0.00014
    },
    "amazon.nova-lite-v1:0": {
        "input": 0.00006,
        "output": 0.00024
    },
    "amazon.nova-pro-v1:0": {
        "input": 0.0008,
        "output": 0.0032
    }
}


test_cases = [
    {
        "question": "What is a 401(k) retirement plan?",
        "ground_truth": (
            "A 401(k) is a tax-advantaged retirement savings "
            "plan offered by employers."
        )
    },
    {
        "question": "What is compound interest?",
        "ground_truth": (
            "Compound interest is interest earned on both the "
            "original principal and previously earned interest."
        )
    },
    {
        "question": (
            "What is the difference between a checking account "
            "and a savings account?"
        ),
        "ground_truth": (
            "A checking account is primarily used for frequent "
            "transactions while a savings account is designed "
            "for storing money and earning interest."
        )
    }
]

In [21]:
def invoke_model(model_id, prompt):
    """
    Sends a prompt to the selected Amazon Bedrock model and
    returns the generated response along with latency and token usage.
    """

    start_time = time.time()

    try:
        response = bedrock.converse(
            modelId=model_id,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "text": prompt
                        }
                    ]
                }
            ],
            inferenceConfig={
                "maxTokens": 150,
                "temperature": 0.2
            }
        )

        latency = time.time() - start_time

        output = response["output"]["message"]["content"][0]["text"]

        usage = response.get("usage", {})

        return {
            "success": True,
            "output": output,
            "latency": latency,
            "input_tokens": usage.get("inputTokens", 0),
            "output_tokens": usage.get("outputTokens", 0)
        }

    except Exception as e:
        return {
            "success": False,
            "error": str(e),
            "latency": time.time() - start_time
        }


In [22]:
test_model_id = "amazon.nova-micro-v1:0"

test_prompt = (
    "Answer the following financial question clearly and accurately "
    "in no more than 3 sentences.\n\n"
    "Question: What is a 401(k) retirement plan?"
)

test_response = invoke_model(
    test_model_id,
    test_prompt
)

test_response

{'success': True,
 'output': 'A 401(k) retirement plan is a tax-advantaged savings account offered by employers, allowing employees to contribute a portion of their salary pre-tax, which grows tax-deferred until withdrawal in retirement, often with employer matching contributions to boost savings.',
 'latency': 0.6727728843688965,
 'input_tokens': 31,
 'output_tokens': 55}

In [23]:
def calculate_similarity(output, ground_truth):
    """
    Calculates a simple word-overlap score between the model's
    response and the expected ground-truth answer.
    """

    output_words = set(output.lower().split())
    truth_words = set(ground_truth.lower().split())

    if not truth_words:
        return 0.0

    common_words = output_words.intersection(truth_words)

    return len(common_words) / len(truth_words)


In [24]:
ground_truth = (
    "A 401(k) is a tax-advantaged retirement savings "
    "plan offered by employers."
)

similarity = calculate_similarity(
    test_response["output"],
    ground_truth
)

similarity

0.9

In [25]:
def calculate_cost(model_id, input_tokens, output_tokens):
    """
    Calculates the estimated Bedrock inference cost for one request
    using the model's input and output token pricing.
    """

    pricing = MODEL_PRICING[model_id]

    input_cost = (
        input_tokens / 1000
    ) * pricing["input"]

    output_cost = (
        output_tokens / 1000
    ) * pricing["output"]

    return input_cost + output_cost


In [26]:
cost = calculate_cost(
    test_model_id,
    test_response["input_tokens"],
    test_response["output_tokens"]
)

cost

8.785e-06

In [27]:
def evaluate_models():
    """
    Runs every test question against every model, collects the
    model responses and performance metrics, and returns them
    as a pandas DataFrame.
    """

    results = []

    for test_case in test_cases:

        prompt = (
            "Answer the following financial question clearly and accurately "
            "in no more than 3 sentences.\n\n"
            f"Question: {test_case['question']}"
        )

        for model_id in models:

            print(
                f"\nEvaluating {model_id}"
                f"\nQuestion: {test_case['question']}"
            )

            response = invoke_model(
                model_id,
                prompt
            )

            if response["success"]:
                cost = calculate_cost(
                    model_id,
                    response["input_tokens"],
                    response["output_tokens"]
                )
                similarity = calculate_similarity(
                    response["output"],
                    test_case["ground_truth"]
                )

                results.append({
                    "model_id": model_id,
                    "question": test_case["question"],
                    "output": response["output"],
                    "latency": response["latency"],
                    "input_tokens": response["input_tokens"],
                    "output_tokens": response["output_tokens"],
                    "similarity_score": similarity,
                    "cost": cost
                })

            else:

                results.append({
                    "model_id": model_id,
                    "question": test_case["question"],
                    "error": response["error"],
                    "latency": response["latency"]
                })

    return pd.DataFrame(results)



In [28]:
results_df = evaluate_models()

results_df


Evaluating amazon.nova-micro-v1:0
Question: What is a 401(k) retirement plan?

Evaluating amazon.nova-lite-v1:0
Question: What is a 401(k) retirement plan?

Evaluating amazon.nova-pro-v1:0
Question: What is a 401(k) retirement plan?

Evaluating amazon.nova-micro-v1:0
Question: What is compound interest?

Evaluating amazon.nova-lite-v1:0
Question: What is compound interest?

Evaluating amazon.nova-pro-v1:0
Question: What is compound interest?

Evaluating amazon.nova-micro-v1:0
Question: What is the difference between a checking account and a savings account?

Evaluating amazon.nova-lite-v1:0
Question: What is the difference between a checking account and a savings account?

Evaluating amazon.nova-pro-v1:0
Question: What is the difference between a checking account and a savings account?


,model_id,question,output,latency,input_tokens,output_tokens,similarity_score,cost
0,amazon.nova-micro-v1:0,What is a 401(k) retirement plan?,A 401(k) retirement plan is a tax-advantaged s...,0.544042,31,74,0.900000,0.000011
1,amazon.nova-lite-v1:0,What is a 401(k) retirement plan?,A 401(k) is an employer-sponsored retirement s...,0.521649,31,43,0.600000,0.000012
2,amazon.nova-pro-v1:0,What is a 401(k) retirement plan?,A 401(k) is a tax-advantaged retirement saving...,0.548100,31,46,0.900000,0.000172
3,amazon.nova-micro-v1:0,What is compound interest?,Compound interest is the interest calculated o...,0.480469,23,52,0.583333,0.000008
4,amazon.nova-lite-v1:0,What is compound interest?,Compound interest is the interest calculated o...,0.458100,23,37,0.500000,0.000010
5,amazon.nova-pro-v1:0,What is compound interest?,Compound interest is interest calculated on th...,0.819291,23,32,0.583333,0.000121
6,amazon.nova-micro-v1:0,What is the difference between a checking acco...,A checking account is primarily used for daily...,0.476062,31,51,0.823529,0.000008
7,amazon.nova-lite-v1:0,What is the difference between a checking acco...,A checking account is primarily used for daily...,0.354934,31,31,0.647059,0.000009
8,amazon.nova-pro-v1:0,What is the difference between a checking acco...,A checking account is designed for frequent tr...,0.850413,31,43,0.764706,0.000162


In [29]:
def create_evaluation_summary(results_df):
    """
    Groups benchmark results by model and calculates the average
    performance metrics for each model.
    """

    summary = results_df.groupby("model_id").agg({
        "latency": "mean",
        "similarity_score": "mean",
        "input_tokens": "mean",
        "output_tokens": "mean",
        "cost": "mean"
    }).reset_index()

    return summary

In [30]:
summary_df = create_evaluation_summary(
    results_df
)

summary_df

,model_id,latency,similarity_score,input_tokens,output_tokens,cost
0,amazon.nova-lite-v1:0,0.444894,0.582353,28.333333,37.000000,0.000011
1,amazon.nova-micro-v1:0,0.500191,0.768954,28.333333,59.000000,0.000009
2,amazon.nova-pro-v1:0,0.739268,0.749346,28.333333,40.333333,0.000152


In [31]:
def measure_throughput(model_id, prompt, request_count=5):
    """
    Sends multiple requests to the same model concurrently and
    calculates how many successful requests are completed per second.
    """

    start_time = time.time()

    with ThreadPoolExecutor(max_workers=request_count) as executor:

        futures = [
            executor.submit(
                invoke_model,
                model_id,
                prompt
            )
            for _ in range(request_count)
        ]

        responses = [
            future.result()
            for future in futures
        ]

    total_time = time.time() - start_time

    successful_requests = sum(
        1
        for response in responses
        if response["success"]
    )

    requests_per_second = (
        successful_requests / total_time
        if total_time > 0
        else 0
    )

    return {
        "model_id": model_id,
        "successful_requests": successful_requests,
        "total_requests": request_count,
        "total_time": total_time,
        "requests_per_second": requests_per_second
    }

Call throughput for Nova Micro

In [32]:
throughput_prompt = (
    "Answer the following financial question clearly and accurately "
    "in no more than 3 sentences.\n\n"
    "Question: What is a 401(k) retirement plan?"
)

micro_throughput = measure_throughput(
    "amazon.nova-micro-v1:0",
    throughput_prompt
)

micro_throughput

{'model_id': 'amazon.nova-micro-v1:0',
 'successful_requests': 5,
 'total_requests': 5,
 'total_time': 0.675307035446167,
 'requests_per_second': 7.404039551722665}

Call throughput for Nova Lite

In [33]:
lite_throughput = measure_throughput(
    "amazon.nova-lite-v1:0",
    throughput_prompt
)

lite_throughput

{'model_id': 'amazon.nova-lite-v1:0',
 'successful_requests': 5,
 'total_requests': 5,
 'total_time': 0.5182619094848633,
 'requests_per_second': 9.64763164819473}

Call throughput for Nova Pro

In [34]:
pro_throughput = measure_throughput(
    "amazon.nova-pro-v1:0",
    throughput_prompt
)

pro_throughput

{'model_id': 'amazon.nova-pro-v1:0',
 'successful_requests': 5,
 'total_requests': 5,
 'total_time': 0.7572898864746094,
 'requests_per_second': 6.602491449181187}

Combine throughput results into a table

In [35]:
throughput_df = pd.DataFrame([
    micro_throughput,
    lite_throughput,
    pro_throughput
])

throughput_df

,model_id,successful_requests,total_requests,total_time,requests_per_second
0,amazon.nova-micro-v1:0,5,5,0.675307,7.404040
1,amazon.nova-lite-v1:0,5,5,0.518262,9.647632
2,amazon.nova-pro-v1:0,5,5,0.757290,6.602491


## Combined decision table

Quality, latency, token usage, cost and throughput are measured separately above.

This section merges them into a single per-model table so the model choice can be
made by reading one table instead of comparing several, and exports it to CSV.

In [36]:
def create_decision_table(summary_df, throughput_df):
    """
    Merges the quality, latency and cost summary with the throughput
    results into a single per-model table that can be used to make
    the model selection decision.
    """

    decision_df = summary_df.merge(
        throughput_df,
        on="model_id",
        how="outer"
    )

    decision_df["success_rate"] = (
        decision_df["successful_requests"] /
        decision_df["total_requests"]
    )

    # Per-request cost is very small, so also show the cost of 1,000 requests.
    decision_df["cost_per_1000_requests"] = decision_df["cost"] * 1000

    decision_df = decision_df.rename(columns={
        "latency": "avg_latency_seconds",
        "similarity_score": "avg_similarity_score",
        "input_tokens": "avg_input_tokens",
        "output_tokens": "avg_output_tokens",
        "cost": "avg_cost_per_request",
        "total_time": "throughput_total_time"
    })

    columns = [
        "model_id",
        "avg_similarity_score",
        "avg_latency_seconds",
        "requests_per_second",
        "avg_input_tokens",
        "avg_output_tokens",
        "avg_cost_per_request",
        "cost_per_1000_requests",
        "successful_requests",
        "total_requests",
        "success_rate",
        "throughput_total_time"
    ]

    decision_df = decision_df[columns]

    return decision_df.sort_values(
        "avg_similarity_score",
        ascending=False
    ).reset_index(drop=True)

In [37]:
decision_df = create_decision_table(
    summary_df,
    throughput_df
)

decision_df

,model_id,avg_similarity_score,avg_latency_seconds,requests_per_second,avg_input_tokens,avg_output_tokens,avg_cost_per_request,cost_per_1000_requests,successful_requests,total_requests,success_rate,throughput_total_time
0,amazon.nova-micro-v1:0,0.768954,0.500191,7.404040,28.333333,59.000000,0.000009,0.009252,5,5,1.0,0.675307
1,amazon.nova-pro-v1:0,0.749346,0.739268,6.602491,28.333333,40.333333,0.000152,0.151733,5,5,1.0,0.757290
2,amazon.nova-lite-v1:0,0.582353,0.444894,9.647632,28.333333,37.000000,0.000011,0.010580,5,5,1.0,0.518262


In [38]:
# Export the single decision table so the comparison can be shared and reviewed.
decision_table_path = "model_benchmark_decision_table.csv"

decision_df.to_csv(
    decision_table_path,
    index=False
)

print(f"Decision table written to {decision_table_path}")

Decision table written to model_benchmark_decision_table.csv


Nova Micro currently offers the **best overall balance**, with the highest similarity score (0.769), low latency (0.50s), strong throughput (7.40 req/s), and the lowest cost (~$0.009 per 1,000 requests).

Nova Lite delivers the **best performance**, with the lowest latency (0.445s) and highest throughput (9.65 req/s), but its similarity score was lower in this run.

Nova Pro was the slowest and by far the most expensive, without showing a clear quality advantage.

**Current ranking:** Nova Micro for overall value, Nova Lite for speed, Nova Pro only if future testing shows a meaningful quality benefit.

### Test model invocation with gurdrails.

In [42]:
def get_guardrail(guardrail_name):
    """
    Finds a Bedrock Guardrail by name and returns its ID
    and latest published version.
    """
    
    bedrock_client = session.client(
        "bedrock",
        region_name="us-east-1"
    )

    response = bedrock_client.list_guardrails()

    guardrail = next(
        (
            item
            for item in response["guardrails"]
            if item["name"] == guardrail_name
        ),
        None
    )

    if not guardrail:
        raise ValueError(
            f"Guardrail '{guardrail_name}' was not found."
        )

    versions_response = bedrock_client.list_guardrails(
        guardrailIdentifier=guardrail["id"]
    )

    published_versions = [
        item
        for item in versions_response["guardrails"]
        if item["version"] != "DRAFT"
    ]

    if not published_versions:
        raise ValueError(
            f"No published version found for '{guardrail_name}'."
        )

    latest_version = max(
        published_versions,
        key=lambda item: int(item["version"])
    )

    return {
        "guardrail_id": guardrail["id"],
        "guardrail_version": latest_version["version"]
    }

In [44]:
guardrail_config = get_guardrail(
    "financial-assistant-guardrail"
)

guardrail_config

{'guardrail_id': 'zt8fazdfv4bi', 'guardrail_version': '1'}

In [45]:
def invoke_model_with_guardrail(
    model_id,
    prompt,
    guardrail_id,
    guardrail_version
):
    """
    Sends a prompt to the selected Amazon Bedrock model with
    a Bedrock Guardrail applied and returns the model response,
    latency, token usage, and guardrail-related information.
    """

    start_time = time.time()

    try:
        response = bedrock.converse(
            modelId=model_id,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "text": prompt
                        }
                    ]
                }
            ],
            inferenceConfig={
                "maxTokens": 150,
                "temperature": 0.2
            },
            guardrailConfig={
                "guardrailIdentifier": guardrail_id,
                "guardrailVersion": guardrail_version,
                "trace": "enabled"
            }
        )

        latency = time.time() - start_time

        output = response["output"]["message"]["content"][0]["text"]

        usage = response.get("usage", {})

        return {
            "success": True,
            "output": output,
            "latency": latency,
            "input_tokens": usage.get("inputTokens", 0),
            "output_tokens": usage.get("outputTokens", 0),
            "stop_reason": response.get("stopReason"),
            "trace": response.get("trace")
        }

    except Exception as e:
        return {
            "success": False,
            "error": str(e),
            "latency": time.time() - start_time
        }

In [46]:
guardrail_response = invoke_model_with_guardrail(
    model_id="amazon.nova-micro-v1:0",
    prompt="What is a 401(k) retirement plan?",
    guardrail_id=guardrail_config["guardrail_id"],
    guardrail_version=guardrail_config["guardrail_version"]
)

guardrail_response

{'success': True,
 'output': "A 401(k) retirement plan is a type of retirement savings account offered by many employers in the United States. It allows employees to save and invest a portion of their salary on a pre-tax basis, which can help reduce their taxable income for the year. Here are the key features of a 401(k) plan:\n\n1. **Employer Contributions**: Many employers offer matching contributions, meaning they will match a certain percentage of the employee's contributions up to a specified limit. This is essentially free money that can significantly boost retirement savings.\n\n2. **Tax Advantages**: Contributions to a traditional 401(k) are typically made with pre-tax dollars, which can lower the employee's taxable income",
 'latency': 1.4819860458374023,
 'input_tokens': 13,
 'output_tokens': 150,
 'stop_reason': 'max_tokens',
 'trace': {'guardrail': {'inputAssessment': {'zt8fazdfv4bi': {'invocationMetrics': {'guardrailProcessingLatency': 188,
      'usage': {'topicPolicyUnit

##### test for blocked ssn

In [47]:
blocked_response = invoke_model_with_guardrail(
    model_id="amazon.nova-micro-v1:0",
    prompt="My Social Security number is 123-45-6789. Can you store it for me?",
    guardrail_id=guardrail_config["guardrail_id"],
    guardrail_version=guardrail_config["guardrail_version"]
)

blocked_response

{'success': True,
 'output': "I can't help with that request.",
 'latency': 0.5743141174316406,
 'input_tokens': 0,
 'output_tokens': 0,
 'stop_reason': 'guardrail_intervened',
 'trace': {'guardrail': {'inputAssessment': {'zt8fazdfv4bi': {'sensitiveInformationPolicy': {'piiEntities': [{'match': '123-45-6789',
        'type': 'US_SOCIAL_SECURITY_NUMBER',
        'action': 'BLOCKED',
        'detected': True}]},
     'invocationMetrics': {'guardrailProcessingLatency': 421,
      'usage': {'topicPolicyUnits': 1,
       'contentPolicyUnits': 1,
       'wordPolicyUnits': 0,
       'sensitiveInformationPolicyUnits': 1,
       'sensitiveInformationPolicyFreeUnits': 0,
       'contextualGroundingPolicyUnits': 0,
       'contentPolicyImageUnits': 0,
       'automatedReasoningPolicyUnits': 0,
       'automatedReasoningPolicies': 0},
      'guardrailCoverage': {'textCharacters': {'guarded': 66, 'total': 66}}},
     'appliedGuardrailDetails': {'guardrailId': 'zt8fazdfv4bi',
      'guardrailVersion

#### create model scores and set defined metrics as needed based on business requirements

***example***
To select the best model, the benchmark metrics are converted into normalized scores and combined into one overall score.

The weights below represent the business priorities for this financial-services assistant:

- **Quality: 60%** — most important because responses must be accurate and useful.
- **Latency: 15%** — important for a responsive customer experience.
- **Throughput: 10%** — measures how many requests the system can process efficiently.
- **Cost: 10%** — helps control inference expenses.
- **Reliability: 5%** — rewards models that successfully complete requests.

These percentages are not fixed AWS values. They are chosen based on business requirements and can be adjusted.

The weights add up to 100%, and the model with the highest overall weighted score becomes the primary candidate.

In [50]:
def create_model_scores(decision_df):
    """
    Calculates normalized scores for quality, latency, throughput,
    cost, and reliability, then combines them into one overall score.
    """

    scores_df = decision_df.copy()

    # Higher similarity is better.
    scores_df["quality_score"] = (
        scores_df["avg_similarity_score"]
    )

    # Lower latency is better.
    max_latency = scores_df["avg_latency_seconds"].max()

    scores_df["latency_score"] = (
        1 - (
            scores_df["avg_latency_seconds"] /
            max_latency
        )
    )

    # Higher throughput is better.
    max_throughput = scores_df["requests_per_second"].max()

    scores_df["throughput_score"] = (
        scores_df["requests_per_second"] /
        max_throughput
    )

    # Lower cost is better.
    max_cost = scores_df["avg_cost_per_request"].max()

    scores_df["cost_score"] = (
        1 - (
            scores_df["avg_cost_per_request"] /
            max_cost
        )
    )

    # Higher success rate is better.
    scores_df["reliability_score"] = (
        scores_df["success_rate"]
    )

    # Weighted overall score.
    scores_df["overall_score"] = (
        0.50 * scores_df["quality_score"] +
        0.10 * scores_df["latency_score"] +
        0.10 * scores_df["throughput_score"] +
        0.10 * scores_df["cost_score"] +
        0.20 * scores_df["reliability_score"]
    )

    return scores_df.sort_values(
        "overall_score",
        ascending=False
    ).reset_index(drop=True)

In [51]:
model_scores_df = create_model_scores(
    decision_df
)

model_scores_df[
    [
        "model_id",
        "quality_score",
        "latency_score",
        "throughput_score",
        "cost_score",
        "reliability_score",
        "overall_score"
    ]
]

,model_id,quality_score,latency_score,throughput_score,cost_score,reliability_score,overall_score
0,amazon.nova-micro-v1:0,0.768954,0.323397,0.767446,0.939027,1.0,0.787464
1,amazon.nova-lite-v1:0,0.582353,0.398196,1.000000,0.930272,1.0,0.724023
2,amazon.nova-pro-v1:0,0.749346,0.000000,0.684364,0.000000,1.0,0.643110


Based on the weighted scoring model, **Amazon Nova Micro ranks first** with the highest overall score.

- **Nova Micro:** best overall balance, driven by the highest response quality and very low cost.
- **Nova Lite:** strongest latency and throughput, but lower response quality reduced its overall score.
- **Nova Pro:** good quality, but higher latency and significantly higher cost made it the weakest overall option.

**Current primary model candidate: Amazon Nova Micro**

**Fallback order:**
1. Amazon Nova Lite
2. Amazon Nova Pro

### Weighting Observation

Changing the weights from 60% quality / 10% reliability to 50% quality / 20% reliability did not change the model ranking.

All three models achieved a reliability score of 1.0, so increasing the reliability weight adds the same value to every model and does not affect their relative ranking.

The ranking is currently driven primarily by differences in response quality, latency, throughput, and cost.